1) Setup

In [2]:
# !pip install scikit-learn pandas numpy joblib plotly --quiet
# Optional accelerators:
# !pip install xgboost lightgbm catboost --quiet

import os, json, warnings, math
from typing import List, Dict, Tuple, Any, Callable

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.linear_model import PoissonRegressor, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import joblib

# Optional libs
_HAS_XGB = _HAS_LGBM = _HAS_CAT = False
try:
    import xgboost as xgb
    from xgboost.callback import EarlyStopping as XGB_EarlyStopping
    _HAS_XGB = True
    print(f"[xgboost] {xgb.__version__}")
except Exception as e:
    warnings.warn(f"xgboost not available: {e}")

try:
    import lightgbm as lgb
    _HAS_LGBM = True
    print(f"[lightgbm] {lgb.__version__}")
except Exception as e:
    warnings.warn(f"lightgbm not available: {e}")

try:
    from catboost import CatBoostRegressor, CatBoostClassifier, Pool
    _HAS_CAT = True
    print("[catboost] available")
except Exception as e:
    warnings.warn(f"catboost not available: {e}")


[xgboost] 2.1.4


C:\Users\Ovy\AppData\Local\Temp\ipykernel_21860\1067312204.py:38: UserWarning: lightgbm not available: No module named 'lightgbm'
  warnings.warn(f"lightgbm not available: {e}")
C:\Users\Ovy\AppData\Local\Temp\ipykernel_21860\1067312204.py:45: UserWarning: catboost not available: No module named 'catboost'
  warnings.warn(f"catboost not available: {e}")


2) Paths & constants

In [3]:
# <<< EDIT this to your actual history CSV path >>>
HISTORY_CSV = "C:/Users/Ovy/Downloads/project etl/cases_with_thresholds.csv"
OUTDIR = "outputs_nb"; os.makedirs(OUTDIR, exist_ok=True)

RANDOM_STATE = 42
LAGS = [1, 2, 4]
ROLL_WINDOWS = [4]

BASE_NUMERIC = ["Week", "Year", "Precipitation", "MIN", "MAX"]
BASE_CATEG  = ["Health Facility"]          # per-facility only
POTENTIAL_LEAKAGE = {"mean_cases","std_cases","alert_threshold","action_threshold"}


3) Utilities (features, metrics, splits)

In [5]:
def add_seasonal(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    ang = 2*np.pi*(out["Week"].astype(float)/52.0)
    out["week_sin"] = np.sin(ang); out["week_cos"] = np.cos(ang)
    return out

def add_lags(df: pd.DataFrame, targets=("Cases","Deaths")) -> pd.DataFrame:
    out = df.sort_values(["Health Facility","Year","Week"]).copy()
    for t in targets:
        if t not in out: out[t] = np.nan
        for L in LAGS:
            out[f"{t}_lag{L}"] = out.groupby("Health Facility")[t].shift(L)
        for W in ROLL_WINDOWS:
            out[f"{t}_roll{W}"] = (
                out.groupby("Health Facility")[t].shift(1).rolling(W, min_periods=1).mean()
                .reset_index(level=0, drop=True)
            )
    return out

def aggregate_facility_week(df: pd.DataFrame) -> pd.DataFrame:
    agg: Dict[str,str] = {"Cases":"sum","Deaths":"sum"}
    for c in ["Precipitation","MIN","MAX"]:
        if c in df: agg[c] = "mean"
    keys = ["Health Facility","Year","Week"]
    return df.groupby(keys, as_index=False).agg(**{k:(k,v) for k,v in agg.items()})

def get_feature_columns(df: pd.DataFrame, target: str) -> List[str]:
    cols = []
    cols += [c for c in BASE_NUMERIC + BASE_CATEG if c in df.columns]
    cols += [c for c in ["week_sin","week_cos"] if c in df.columns]
    cols += [c for c in df.columns if any(k in c for k in ["Cases_lag","Cases_roll","Deaths_lag","Deaths_roll"])]
    cols = [c for c in cols if c not in POTENTIAL_LEAKAGE and c != target]
    # stable unique
    seen, out = set(), []
    for c in cols:
        if c not in seen: out.append(c); seen.add(c)
    return out

def year_rolling_splits(years_sorted: List[int], min_train_years=2):
    for i in range(min_train_years, len(years_sorted)):
        yield years_sorted[:i], years_sorted[i]

def wape(y_true, y_pred):
    y_true = np.asarray(y_true,float); y_pred = np.asarray(y_pred,float)
    denom = np.sum(np.abs(y_true))
    return np.nan if denom <= 1e-8 else np.sum(np.abs(y_true - y_pred))/denom

def smape(y_true, y_pred):
    y_true = np.asarray(y_true,float); y_pred = np.asarray(y_pred,float)
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    denom = np.where(denom<1e-8, 1.0, denom)
    return float(np.mean(np.abs(y_true-y_pred)/denom))


4) Robust fit wrappers (early stopping where possible)

In [6]:
def xgb_fit_with_eval(model, Xtr, ytr, Xva, yva, metrics=("rmse",), early_rounds=200, verbose=False):
    if not _HAS_XGB: 
        model.fit(Xtr, ytr); return
    try:
        model.set_params(eval_metric=list(metrics))
    except Exception: pass
    try:
        model.fit(Xtr, ytr, eval_set=[(Xtr,ytr),(Xva,yva)],
                  verbose=verbose, callbacks=[XGB_EarlyStopping(rounds=early_rounds, save_best=True)])
        return
    except Exception: ...
    try:
        model.fit(Xtr, ytr, eval_set=[(Xtr,ytr),(Xva,yva)],
                  eval_metric=list(metrics), verbose=verbose, early_stopping_rounds=early_rounds)
        return
    except Exception: ...
    try:
        model.fit(Xtr, ytr, eval_set=[(Xtr,ytr),(Xva,yva)], verbose=verbose); return
    except Exception: ...
    model.fit(Xtr, ytr)

def lgbm_fit_with_eval(model, Xtr, ytr, Xva, yva, metric="rmse"):
    if not _HAS_LGBM: model.fit(Xtr, ytr); return
    model.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric=metric,
        callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
    )

def cat_fit_with_eval(model, Xtr, ytr, Xva, yva, is_classifier=False):
    if not _HAS_CAT: model.fit(Xtr, ytr); return
    model.fit(Xtr, ytr, eval_set=(Xva, yva), verbose=False, use_best_model=True)


5) Candidate algorithms (builders + small grids)

In [7]:
def make_preproc(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ("num","passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False
    )

# === CASES regressors ===
def cases_candidates(feat_cols) -> Dict[str, Tuple[Pipeline, List[Dict[str,Any]], Callable]]:
    num = [c for c in feat_cols if c not in BASE_CATEG]; cat = [c for c in feat_cols if c in BASE_CATEG]
    pre = make_preproc(num, cat)
    cands = {}

    # Poisson (interpretable baseline)
    cands["PoissonRegressor"] = (
        Pipeline([("prep", pre), ("model", PoissonRegressor(alpha=0.0, max_iter=1000))]),
        list(ParameterGrid({"model__alpha":[0.0, 0.001, 0.01]})),
        None
    )

    # Random Forest
    cands["RandomForestRegressor"] = (
        Pipeline([("prep", pre), ("model", RandomForestRegressor(
            n_estimators=400, max_depth=None, min_samples_split=2, min_samples_leaf=1, n_jobs=-1, random_state=RANDOM_STATE))]),
        list(ParameterGrid({"model__n_estimators":[400,800], "model__max_depth":[None,12]})),
        None
    )

    # Gradient Boosting (sklearn)
    cands["GradientBoostingRegressor"] = (
        Pipeline([("prep", pre), ("model", GradientBoostingRegressor(
            n_estimators=1000, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))]),
        list(ParameterGrid({"model__learning_rate":[0.03,0.05,0.08], "model__max_depth":[3,4]})),
        None
    )

    # XGBoost
    if _HAS_XGB:
        cands["XGBRegressor"] = (
            Pipeline([("prep", pre), ("model", xgb.XGBRegressor(
                objective="count:poisson", tree_method="hist", n_estimators=4000,
                learning_rate=0.06, max_depth=6, subsample=0.9, colsample_bytree=0.9,
                random_state=RANDOM_STATE, n_jobs=0))]),
            list(ParameterGrid({
                "model__learning_rate":[0.04,0.06,0.08],
                "model__max_depth":[5,6,8],
                "model__subsample":[0.8,0.9],
                "model__colsample_bytree":[0.8,0.9]
            })),
            xgb_fit_with_eval
        )

    # LightGBM
    if _HAS_LGBM:
        cands["LGBMRegressor"] = (
            Pipeline([("prep", pre), ("model", lgb.LGBMRegressor(
                objective="poisson", n_estimators=4000, learning_rate=0.06, max_depth=-1,
                subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE))]),
            list(ParameterGrid({
                "model__learning_rate":[0.04,0.06,0.08],
                "model__num_leaves":[31,63],
                "model__subsample":[0.8,0.9],
                "model__colsample_bytree":[0.8,0.9]
            })),
            lgbm_fit_with_eval
        )

    # CatBoost
    if _HAS_CAT:
        cands["CatBoostRegressor"] = (
            Pipeline([("prep", pre), ("model", CatBoostRegressor(
                loss_function="RMSE", depth=6, learning_rate=0.06, iterations=4000,
                random_state=RANDOM_STATE, allow_writing_files=False, verbose=False))]),
            list(ParameterGrid({"model__depth":[6,8], "model__learning_rate":[0.04,0.06]})),
            cat_fit_with_eval
        )
    return cands

# === DEATHS hurdle candidates ===
def deaths_classifier_candidates(feat_cols):
    num = [c for c in feat_cols if c not in BASE_CATEG]; cat = [c for c in feat_cols if c in BASE_CATEG]
    pre = make_preproc(num, cat)
    cands = {
        "LogisticRegression": (
            Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=1000, n_jobs=None, solver="lbfgs"))]),
            list(ParameterGrid({"model__C":[0.5,1.0,2.0]})),
            None
        ),
        "RandomForestClassifier": (
            Pipeline([("prep", pre), ("model", RandomForestClassifier(
                n_estimators=500, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE))]),
            list(ParameterGrid({"model__n_estimators":[500,800], "model__max_depth":[None,12]})),
            None
        )
    }
    if _HAS_XGB:
        cands["XGBClassifier"] = (
            Pipeline([("prep", pre), ("model", xgb.XGBClassifier(
                objective="binary:logistic", tree_method="hist", n_estimators=4000,
                learning_rate=0.06, max_depth=5, subsample=0.9, colsample_bytree=0.9,
                random_state=RANDOM_STATE, n_jobs=0))]),
            list(ParameterGrid({
                "model__learning_rate":[0.04,0.06,0.08],
                "model__max_depth":[4,5,6],
                "model__subsample":[0.8,0.9],
                "model__colsample_bytree":[0.8,0.9]
            })),
            xgb_fit_with_eval
        )
    if _HAS_LGBM:
        cands["LGBMClassifier"] = (
            Pipeline([("prep", pre), ("model", lgb.LGBMClassifier(
                objective="binary", n_estimators=4000, learning_rate=0.06,
                subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE))]),
            list(ParameterGrid({
                "model__learning_rate":[0.04,0.06,0.08],
                "model__num_leaves":[31,63],
                "model__subsample":[0.8,0.9],
                "model__colsample_bytree":[0.8,0.9]
            })),
            lgbm_fit_with_eval
        )
    if _HAS_CAT:
        cands["CatBoostClassifier"] = (
            Pipeline([("prep", pre), ("model", CatBoostClassifier(
                loss_function="Logloss", depth=6, learning_rate=0.06, iterations=4000,
                random_state=RANDOM_STATE, allow_writing_files=False, verbose=False))]),
            list(ParameterGrid({"model__depth":[6,8], "model__learning_rate":[0.04,0.06]})),
            cat_fit_with_eval
        )
    return cands

def deaths_regressor_candidates(feat_cols):
    num = [c for c in feat_cols if c not in BASE_CATEG]; cat = [c for c in feat_cols if c in BASE_CATEG]
    pre = make_preproc(num, cat)
    cands = {
        "PoissonRegressor": (
            Pipeline([("prep", pre), ("model", PoissonRegressor(alpha=0.0, max_iter=1000))]),
            list(ParameterGrid({"model__alpha":[0.0, 0.001, 0.01]})),
            None
        ),
        "RandomForestRegressor": (
            Pipeline([("prep", pre), ("model", RandomForestRegressor(
                n_estimators=400, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE))]),
            list(ParameterGrid({"model__n_estimators":[400,800], "model__max_depth":[None,12]})),
            None
        ),
        "GradientBoostingRegressor": (
            Pipeline([("prep", pre), ("model", GradientBoostingRegressor(
                n_estimators=1000, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE))]),
            list(ParameterGrid({"model__learning_rate":[0.03,0.05,0.08], "model__max_depth":[3,4]})),
            None
        )
    }
    if _HAS_XGB:
        cands["XGBRegressor"] = (
            Pipeline([("prep", pre), ("model", xgb.XGBRegressor(
                objective="count:poisson", tree_method="hist", n_estimators=4000,
                learning_rate=0.06, max_depth=5, subsample=0.9, colsample_bytree=0.9,
                random_state=RANDOM_STATE, n_jobs=0))]),
            list(ParameterGrid({
                "model__learning_rate":[0.04,0.06,0.08],
                "model__max_depth":[4,5,6],
                "model__subsample":[0.8,0.9],
                "model__colsample_bytree":[0.8,0.9]
            })),
            xgb_fit_with_eval
        )
    if _HAS_LGBM:
        cands["LGBMRegressor"] = (
            Pipeline([("prep", pre), ("model", lgb.LGBMRegressor(
                objective="poisson", n_estimators=4000, learning_rate=0.06,
                subsample=0.9, colsample_bytree=0.9, random_state=RANDOM_STATE))]),
            list(ParameterGrid({
                "model__learning_rate":[0.04,0.06,0.08],
                "model__num_leaves":[31,63],
                "model__subsample":[0.8,0.9],
                "model__colsample_bytree":[0.8,0.9]
            })),
            lgbm_fit_with_eval
        )
    if _HAS_CAT:
        cands["CatBoostRegressor"] = (
            Pipeline([("prep", pre), ("model", CatBoostRegressor(
                loss_function="RMSE", depth=6, learning_rate=0.06, iterations=4000,
                random_state=RANDOM_STATE, allow_writing_files=False, verbose=False))]),
            list(ParameterGrid({"model__depth":[6,8], "model__learning_rate":[0.04,0.06]})),
            cat_fit_with_eval
        )
    return cands


6) Load, clean, aggregate, feature-engineer

In [8]:
# %% [markdown]
# ### Cell 6 — Load/prepare data + robust preprocessing (imputers + OneHot)

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ---------- Version-safe OneHot and preprocessing builders ----------
def _onehot_encoder():
    """
    Construct a OneHotEncoder that works across scikit-learn versions:
    - sklearn >= 1.2: use sparse_output
    - older sklearn:  use sparse
    """
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        # Fallback for older sklearn versions
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preproc(num_cols, cat_cols):
    """
    ColumnTransformer with:
      - Numeric: median imputation
      - Categorical: most-frequent imputation + OneHot
    """
    num_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="median")),
        # If you ever need scaling for linear models, you can add it here:
        # ("scale", StandardScaler())
    ])

    cat_pipe = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", _onehot_encoder()),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return pre

def build_preproc_for(feat_cols, base_categ=None):
    """
    Split feature list into numeric & categorical, then return the preprocessor.
    By default, treat 'Health Facility' as categorical key.
    """
    if base_categ is None:
        base_categ = ["Health Facility"]

    cat_cols = [c for c in feat_cols if c in base_categ]
    num_cols = [c for c in feat_cols if c not in cat_cols]
    pre = make_preproc(num_cols, cat_cols)
    return pre, num_cols, cat_cols

# ---------- Load & feature engineering (your existing steps, kept) ----------
raw = pd.read_csv(HISTORY_CSV)
raw.columns = [c.strip() for c in raw.columns]

# Coerce numeric columns safely (only if present)
for c in ["Year","Week","Cases","Deaths","Precipitation","MIN","MAX"]:
    if c in raw.columns:
        raw[c] = pd.to_numeric(raw[c], errors="coerce")

# Aggregate per Health Facility × Week (your helper should already exist)
df = aggregate_facility_week(raw)          # <- keeps "Health Facility" level (no Station duplication)
df = add_seasonal(df)
df = add_lags(df, targets=("Cases","Deaths"))

# Split cases/deaths modelling frames (drop rows with missing target only)
cases_df  = df.dropna(subset=["Cases"]).copy()
deaths_df = df.dropna(subset=["Deaths"]).copy()

years_cases  = sorted(cases_df["Year"].dropna().unique().tolist())
years_deaths = sorted(deaths_df["Year"].dropna().unique().tolist())

# Pick features (your existing helper)
feat_cols_c = get_feature_columns(cases_df,  target="Cases")
feat_cols_d = get_feature_columns(deaths_df, target="Deaths")

# Build robust preprocessors (median impute numeric, most-frequent+onehot for 'Health Facility')
pre_c, num_cols_c, cat_cols_c = build_preproc_for(feat_cols_c, base_categ=["Health Facility"])
pre_d, num_cols_d, cat_cols_d = build_preproc_for(feat_cols_d, base_categ=["Health Facility"])

print(f"Cases rows: {len(cases_df):,}  Years: {years_cases}  |  Features: {len(feat_cols_c)} "
      f"(num={len(num_cols_c)}, cat={len(cat_cols_c)})")
print(f"Deaths rows: {len(deaths_df):,} Years: {years_deaths} |  Features: {len(feat_cols_d)} "
      f"(num={len(num_cols_d)}, cat={len(cat_cols_d)})")

# Optional quick sanity check: ensure the preprocessor can transform without NaN errors.
# (Comment out if you prefer to skip this quick test.)
try:
    _ = pre_c.fit_transform(cases_df[feat_cols_c])
    _ = pre_d.fit_transform(deaths_df[feat_cols_d])
    print("✅ Preprocessors fitted & transformed successfully (no NaN leakage to estimators).")
except Exception as e:
    print("⚠️ Preprocessor transform check failed:", e)


Cases rows: 8,679  Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  |  Features: 16 (num=15, cat=1)
Deaths rows: 8,679 Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024] |  Features: 16 (num=15, cat=1)
✅ Preprocessors fitted & transformed successfully (no NaN leakage to estimators).


7) Train & compare — CASES (auto-select)

In [9]:
 # %% [markdown]
# # CASES — Auto-tuning with interactive logs (loops until WAPE < 0.40)

# %%
import os, json, time, math, warnings, joblib, random
import numpy as np
import pandas as pd
from copy import deepcopy
from typing import Dict, Any, List, Tuple

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor

# Optional boosters
_HAS_XGB = _HAS_LGBM = False
try:
    import xgboost as xgb
    _HAS_XGB = True
except Exception:
    pass
try:
    import lightgbm as lgb
    _HAS_LGBM = True
except Exception:
    pass

def log(*a, **k):
    """Simple timestamped logger."""
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}]", *a, **k, flush=True)

# --- Safety: year_rolling_splits fallback
def _year_splits_fallback(years: List[int], min_train_years: int = 2):
    ys = sorted(list(years))
    for i in range(min_train_years, len(ys)):
        tr = ys[:i]
        va = ys[i]
        if len(tr) >= min_train_years:
            yield tr, va
if "year_rolling_splits" not in globals():
    year_rolling_splits = _year_splits_fallback

# --- Metrics
def wape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = np.maximum(1.0, np.abs(y_true)).sum()
    return float(np.abs(y_true - y_pred).sum() / denom)

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    num = np.abs(y_pred - y_true)
    den = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    den = np.where(den == 0, 1.0, den)
    return float(np.mean(num / den))

def mase(y_true, y_pred, y_naive):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    y_naive = np.asarray(y_naive, float)
    num = np.mean(np.abs(y_true - y_pred))
    den = np.mean(np.abs(y_true - y_naive)) if np.any(np.isfinite(y_naive)) else np.nan
    if not np.isfinite(den) or den == 0:
        return np.nan
    return float(num / den)

# Seasonal-naive baseline (same Week last year with fallbacks)
def seasonal_naive_for_fold(valid_df: pd.DataFrame, train_df: pd.DataFrame) -> np.ndarray:
    tr = train_df[["Health Facility","Year","Week","Cases"]].copy()
    va = valid_df[["Health Facility","Year","Week"]].copy()
    last_year = va["Year"].min() - 1

    sw = va.merge(tr[tr["Year"]==last_year][["Health Facility","Week","Cases"]],
                  on=["Health Facility","Week"], how="left")["Cases"].to_numpy()
    need = ~np.isfinite(sw)
    if need.any():
        va_pw = va.loc[need].copy()
        va_pw["Week"] = np.clip(va_pw["Week"] - 1, 1, 52)
        pw = va_pw.merge(tr[tr["Year"]==last_year][["Health Facility","Week","Cases"]],
                         on=["Health Facility","Week"], how="left")["Cases"].to_numpy()
        sw = np.where(need, pw, sw)
    need2 = ~np.isfinite(sw)
    if need2.any():
        med = (tr.groupby("Health Facility")["Cases"].median()
                 .reindex(va.loc[need2,"Health Facility"]).to_numpy())
        sw = np.where(need2, med, sw)
    return np.where(np.isfinite(sw), sw, 0.0)

# --- Preprocessing (drop Station to avoid station-level duplication)
num_cols = [c for c in feat_cols_c if c not in ["Health Facility","Station"]]
cat_cols = [c for c in feat_cols_c if c in ["Health Facility"]]

numeric_prep = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_prep = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_prep, num_cols),
    ("cat", categorical_prep, cat_cols),
], remainder="drop")

def make_log1p_pipe(base_estimator):
    return Pipeline([
        ("prep", preprocessor),
        ("model", TransformedTargetRegressor(
            regressor=base_estimator,
            func=np.log1p, inverse_func=np.expm1
        ))
    ])

# --- Candidates & seed grids
def cases_candidates_refined() -> Dict[str, Tuple[Pipeline, List[Dict[str, Any]]]]:
    cands = {}

    gbr = GradientBoostingRegressor(random_state=42)
    cands["GBR_log1p"] = (
        make_log1p_pipe(gbr),
        list(ParameterGrid({
            "model__regressor__n_estimators": [600, 1000, 1400],
            "model__regressor__learning_rate": [0.03, 0.06],
            "model__regressor__max_depth": [2, 3],
            "model__regressor__subsample": [0.9, 1.0],
        }))
    )

    rf = RandomForestRegressor(random_state=42, n_jobs=-1)
    cands["RF_log1p"] = (
        make_log1p_pipe(rf),
        list(ParameterGrid({
            "model__regressor__n_estimators": [600, 1000, 1400],
            "model__regressor__max_depth": [None, 12],
            "model__regressor__min_samples_leaf": [1, 2],
            "model__regressor__max_features": ["sqrt", 0.6],
        }))
    )

    if _HAS_LGBM:
        lgbm = lgb.LGBMRegressor(objective="poisson", random_state=42, n_jobs=-1, verbosity=-1)
        cands["LGBM_Poisson"] = (
            Pipeline([("prep", preprocessor), ("model", lgbm)]),
            list(ParameterGrid({
                "model__n_estimators": [1200, 1800],
                "model__learning_rate": [0.03, 0.06],
                "model__num_leaves": [31, 63],
                "model__min_data_in_leaf": [20, 60],
                "model__subsample": [0.8, 1.0],
                "model__colsample_bytree": [0.8, 1.0],
            }))
        )

    if _HAS_XGB:
        xgbr = xgb.XGBRegressor(
            random_state=42, n_estimators=1600, tree_method="hist",
            n_jobs=-1, reg_alpha=0.0, reg_lambda=1.0, verbosity=0
        )
        cands["XGB_log1p"] = (
            make_log1p_pipe(xgbr),
            list(ParameterGrid({
                "model__regressor__learning_rate": [0.03, 0.06],
                "model__regressor__max_depth": [3, 5],
                "model__regressor__subsample": [0.8, 1.0],
                "model__regressor__colsample_bytree": [0.8, 1.0],
            }))
        )

    try:
        from sklearn.linear_model import TweedieRegressor
        tr = TweedieRegressor(power=1.3, alpha=0.0, link="log", max_iter=1000)
        cands["Tweedie_raw"] = (
            Pipeline([("prep", preprocessor), ("model", tr)]),
            list(ParameterGrid({
                "model__power": [1.1, 1.3, 1.5],
                "model__alpha": [0.0, 0.001, 0.01]
            }))
        )
    except Exception:
        pass

    return cands

years_cases = sorted(cases_df["Year"].dropna().unique().tolist())

def _extract_booster_fold_loss(fitted_pipe):
    """Try to read last eval metric from boosters inside TTR where possible."""
    # TTR is under step 'model'; inside it, after fit, attribute is regressor_
    try:
        ttr = fitted_pipe.named_steps["model"]
        base = getattr(ttr, "regressor_", None) or getattr(ttr, "regressor", None)
        # LightGBM
        if _HAS_LGBM and isinstance(base, lgb.LGBMRegressor):
            try:
                # LGBM doesn't auto-store evals by default; we pass eval_set in fit below only for boosters (see evaluate)
                # If available:
                res = base.booster_.current_iteration()
                return f"lgb_iter={res}"
            except Exception:
                return None
        # XGBoost
        if _HAS_XGB and isinstance(base, xgb.XGBRegressor):
            try:
                # xgb stores evals_result_ when eval_set is provided
                er = base.evals_result()
                if not er: return None
                # choose last valid metric on eval set 1 (validation)
                eval_set_name = [k for k in er.keys() if "validation_1" in er[k]][0] if isinstance(er, dict) else "validation_1"
                metrics = list(er[eval_set_name].keys())
                m = metrics[0]
                last = er[eval_set_name][m][-1]
                return f"xgb_{m}={last:.4f}"
            except Exception:
                return None
    except Exception:
        pass
    return None

def evaluate_cases_model(pipe: Pipeline, params: Dict[str,Any], *, verbose: bool = True) -> Dict[str, float]:
    pipe = deepcopy(pipe)
    pipe.set_params(**params)
    fold_metrics = []
    min_train_years = max(2, len(years_cases)//2) if len(years_cases) >= 3 else 1
    fnum = 0

    for tr_years, va_year in year_rolling_splits(years_cases, min_train_years=min_train_years):
        fnum += 1
        tr = cases_df[cases_df["Year"].isin(tr_years)]
        va = cases_df[cases_df["Year"] == va_year]
        Xtr, ytr = tr[feat_cols_c], tr["Cases"].values
        Xva, yva = va[feat_cols_c], va["Cases"].values

        fitted = None
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # If booster, try to pass eval_set for logs (only for the inner estimator)
            is_xgb = _HAS_XGB and ("XGB" in "".join([type(est).__name__ for est in pipe.steps]))
            is_lgb = _HAS_LGBM and ("LGBM" in "".join([type(est).__name__ for est in pipe.steps]))

            if is_xgb or is_lgb:
                # split the pipe to inject eval_set: fit preprocessor first
                pre = deepcopy(pipe.named_steps["prep"])
                Xtr_t = pre.fit_transform(Xtr)
                Xva_t = pre.transform(Xva)

                # find TransformedTargetRegressor
                ttr = deepcopy(pipe.named_steps["model"])
                base = ttr.regressor

                if is_xgb and isinstance(base, xgb.XGBRegressor):
                    try:
                        base.set_params(early_stopping_rounds=100, eval_metric="rmse", verbosity=0)
                    except Exception:
                        pass
                    base.fit(
                        Xtr_t, np.log1p(ytr),
                        eval_set=[(Xtr_t, np.log1p(ytr)), (Xva_t, np.log1p(yva))],
                        verbose=False
                    )
                    # rebuild the pipe with fitted pre + TTR (wrap back)
                    ttr_fitted = TransformedTargetRegressor(
                        regressor=base, func=np.log1p, inverse_func=np.expm1
                    )
                    # mimic fitted state
                    ttr_fitted.regressor_ = base
                    fitted = Pipeline([("prep", pre), ("model", ttr_fitted)])

                elif is_lgb and isinstance(base, lgb.LGBMRegressor):
                    try:
                        base.set_params(metric="rmse")
                    except Exception:
                        pass
                    base.fit(
                        Xtr_t, ytr,
                        eval_set=[(Xtr_t, ytr), (Xva_t, yva)],
                        eval_metric="rmse",
                        verbose=False
                    )
                    ttr_fitted = TransformedTargetRegressor(
                        regressor=base, func=np.log1p, inverse_func=np.expm1
                    )
                    ttr_fitted.regressor_ = base
                    fitted = Pipeline([("prep", pre), ("model", ttr_fitted)])
                else:
                    fitted = pipe.fit(Xtr, ytr)
            else:
                fitted = pipe.fit(Xtr, ytr)

        yhat = np.clip(fitted.predict(Xva), 0, None)
        y_naive = seasonal_naive_for_fold(va, tr)

        fold_wape = wape(yva, yhat)
        fold_smape = smape(yva, yhat)
        fold_rmse = float(np.sqrt(mean_squared_error(yva, yhat)))
        fold_mase = mase(yva, yhat, y_naive)

        booster_loss = _extract_booster_fold_loss(fitted)
        if verbose:
            log(f"  Fold {fnum} | Year {va_year}  WAPE={fold_wape:.4f}  SMAPE={fold_smape:.4f}  RMSE={fold_rmse:.3f}  MASE={fold_mase:.3f}"
                + (f"  ({booster_loss})" if booster_loss else ""))

        fold_metrics.append({
            "WAPE": fold_wape, "SMAPE": fold_smape,
            "RMSE": fold_rmse, "MASE": fold_mase
        })

    m = pd.DataFrame(fold_metrics).mean(numeric_only=True).to_dict()
    for k in ("WAPE","SMAPE","RMSE","MASE"):
        if k not in m or not np.isfinite(m[k]): m[k] = np.inf
    return m

# --- Refinement: random local search around current best params
def _perturb(val, scale=0.2, minv=None, maxv=None, dtype=float):
    if isinstance(val, (int, np.integer)):
        span = max(1, int(abs(val)*scale))
        cand = int(max(1, val + random.randint(-span, span)))
        if minv is not None: cand = max(minv, cand)
        if maxv is not None: cand = min(maxv, cand)
        return cand
    if isinstance(val, (float, np.floating)):
        cand = float(val) * (1.0 + random.uniform(-scale, scale))
        if minv is not None: cand = max(minv, cand)
        if maxv is not None: cand = min(maxv, cand)
        return dtype(cand)
    return val  # categorical

def refine_grid_locally(model_name: str, best_params: Dict[str, Any], n_new: int = 10) -> List[Dict[str, Any]]:
    """Create small random neighbors around best_params for a given model family."""
    new = []
    for _ in range(n_new):
        p = deepcopy(best_params)
        if model_name == "GBR_log1p":
            for k in ("model__regressor__n_estimators","model__regressor__max_depth"):
                if k in p: p[k] = _perturb(p[k], scale=0.3, minv=1)
            if "model__regressor__learning_rate" in p:
                p["model__regressor__learning_rate"] = _perturb(p["model__regressor__learning_rate"], scale=0.3, minv=0.005, maxv=0.2, dtype=float)
            if "model__regressor__subsample" in p:
                p["model__regressor__subsample"] = min(1.0, max(0.5, _perturb(p["model__regressor__subsample"], scale=0.2, dtype=float)))
        elif model_name == "RF_log1p":
            for k in ("model__regressor__n_estimators",):
                if k in p: p[k] = _perturb(p[k], scale=0.3, minv=100)
            # max_features tweak
            if "model__regressor__max_features" in p and isinstance(p["model__regressor__max_features"], float):
                p["model__regressor__max_features"] = float(min(1.0, max(0.2, _perturb(p["model__regressor__max_features"], scale=0.25))))
        elif model_name == "LGBM_Poisson":
            for k in ("model__n_estimators","model__num_leaves","model__min_data_in_leaf"):
                if k in p: p[k] = _perturb(p[k], scale=0.3, minv=5)
            if "model__learning_rate" in p:
                p["model__learning_rate"] = _perturb(p["model__learning_rate"], scale=0.3, minv=0.005, maxv=0.2, dtype=float)
            for k in ("model__subsample","model__colsample_bytree"):
                if k in p: p[k] = float(min(1.0, max(0.5, _perturb(p[k], scale=0.25))))
        elif model_name == "XGB_log1p":
            for k in ("model__regressor__n_estimators","model__regressor__max_depth"):
                if k in p: p[k] = _perturb(p[k], scale=0.3, minv=1)
            if "model__regressor__learning_rate" in p:
                p["model__regressor__learning_rate"] = _perturb(p["model__regressor__learning_rate"], scale=0.3, minv=0.005, maxv=0.2, dtype=float)
            for k in ("model__regressor__subsample","model__regressor__colsample_bytree"):
                if k in p: p[k] = float(min(1.0, max(0.5, _perturb(p[k], scale=0.25))))
        elif model_name == "Tweedie_raw":
            if "model__power" in p:
                p["model__power"] = float(min(1.8, max(1.0, _perturb(p["model__power"], scale=0.1))))
            if "model__alpha" in p:
                p["model__alpha"] = float(min(0.1, max(0.0, _perturb(p["model__alpha"], scale=0.5))))
        new.append(p)
    return new

# --- Search, log, refine until WAPE < 0.40 (or stop)
TARGET_WAPE = 0.40
MAX_ROUNDS = 4            # safety cap of refinement rounds
MAX_TRIALS_PER_MODEL = 12 # per round

cand = cases_candidates_refined()
all_results = []  # accumulate across rounds
best_global = (np.inf, None, None)  # (WAPE, model_name, params)

for rnd in range(1, MAX_ROUNDS+1):
    log(f"=== ROUND {rnd} / {MAX_ROUNDS} — Searching candidates ===")
    round_results = []

    for name, (pipe, grid) in cand.items():
        # choose a subset for speed per round
        chosen = random.sample(grid, k=min(len(grid), MAX_TRIALS_PER_MODEL))
        log(f"→ {name}: trying {len(chosen)} combinations")
        for i, params in enumerate(chosen, 1):
            log(f"  [{name}] trial {i}/{len(chosen)} params={params}")
            met = evaluate_cases_model(pipe, params, verbose=True)
            rec = {"round": rnd, "model": name, "params": params, **met}
            round_results.append(rec)
            all_results.append(rec)
            log(f"  [{name}] → WAPE={met['WAPE']:.4f}  SMAPE={met['SMAPE']:.4f}  MASE={met['MASE']:.4f}  RMSE={met['RMSE']:.3f}")

    # summarize round & check threshold
    dfr = pd.DataFrame(round_results).sort_values(["WAPE","SMAPE","MASE"], ascending=[True,True,True])
    if dfr.empty:
        log("No results this round — stopping.")
        break
    best_row = dfr.iloc[0]
    log(f"=== ROUND {rnd} best: {best_row['model']}  WAPE={best_row['WAPE']:.4f} (SMAPE={best_row['SMAPE']:.4f}, MASE={best_row['MASE']:.4f})")

    if best_row["WAPE"] < best_global[0]:
        best_global = (float(best_row["WAPE"]), best_row["model"], best_row["params"])

    if best_global[0] < TARGET_WAPE:
        log(f"Target reached (WAPE {best_global[0]:.4f} < {TARGET_WAPE}). Stopping search ✅")
        break

    # build refined grids around each model's current round best
    log("Refining grids around round-best per model …")
    refined = {}
    for name in cand.keys():
        sub = dfr[dfr["model"]==name]
        if sub.empty: 
            continue
        best_p = sub.iloc[0]["params"]
        refined[name] = (cand[name][0], refine_grid_locally(name, best_p, n_new=8))
        log(f"  {name}: generated {len(refined[name][1])} local neighbors")
    cand.update(refined)

# Final selection across all rounds
df_all = pd.DataFrame(all_results).sort_values(["WAPE","SMAPE","MASE"], ascending=[True,True,True]).reset_index(drop=True)
if df_all.empty:
    raise RuntimeError("No successful model fits; check data and features.")

best_cases_name   = df_all.loc[0, "model"]
best_cases_params = df_all.loc[0, "params"]
best_cases_wape   = float(df_all.loc[0, "WAPE"])
best_cases_smape  = float(df_all.loc[0, "SMAPE"])
best_cases_mase   = float(df_all.loc[0, "MASE"])
best_cases_rmse   = float(df_all.loc[0, "RMSE"])

log("=== GLOBAL WINNER ===")
log(f"Model: {best_cases_name}")
log(f"WAPE={best_cases_wape:.4f}  SMAPE={best_cases_smape:.4f}  MASE={best_cases_mase:.4f}  RMSE={best_cases_rmse:.3f}")
log(f"Params: {best_cases_params}")

# Fit on ALL data and SAVE
cands_seed = cases_candidates_refined()  # fresh instance of candidate objects
best_pipe_cases = cands_seed[best_cases_name][0]
best_pipe_cases.set_params(**best_cases_params)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    best_pipe_cases.fit(cases_df[feat_cols_c], cases_df["Cases"])

os.makedirs(OUTDIR, exist_ok=True)
joblib.dump(best_pipe_cases, os.path.join(OUTDIR, "best_cases_pipeline.joblib"))
log("[SAVED] best_cases_pipeline.joblib")

# Update / create training_summary.json
summary_path = os.path.join(OUTDIR, "training_summary.json")
summary_blob = {}
if os.path.exists(summary_path):
    try:
        with open(summary_path, "r", encoding="utf-8") as f:
            summary_blob = json.load(f)
    except Exception:
        summary_blob = {}

summary_blob["cases"] = {
    "winner_model": best_cases_name,
    "winner_params": best_cases_params,
    "metric_primary": "WAPE",
    "scores": {
        "WAPE": best_cases_wape,
        "SMAPE": best_cases_smape,
        "MASE": best_cases_mase,
        "RMSE": best_cases_rmse
    },
    "target_WAPE": TARGET_WAPE,
    "rounds_run": int(min(MAX_ROUNDS, df_all["round"].max())),
    "trials": int(len(df_all))
}
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_blob, f, indent=2)
log("[SAVED] training_summary.json (cases section updated)")

# Optional: quick table view at the end
log("Top 15 candidates overall (by WAPE):")
display(df_all.head(15)[["model","WAPE","SMAPE","MASE","RMSE","params"]])


[09:33:53] === ROUND 1 / 4 — Searching candidates ===
[09:33:53] → GBR_log1p: trying 12 combinations
[09:33:53]   [GBR_log1p] trial 1/12 params={'model__regressor__learning_rate': 0.06, 'model__regressor__max_depth': 2, 'model__regressor__n_estimators': 600, 'model__regressor__subsample': 0.9}
[09:33:56]   Fold 1 | Year 2019  WAPE=0.4990  SMAPE=0.9530  RMSE=25.371  MASE=0.694
[09:33:58]   Fold 2 | Year 2020  WAPE=0.6542  SMAPE=0.8643  RMSE=42.223  MASE=0.857
[09:34:01]   Fold 3 | Year 2021  WAPE=0.5428  SMAPE=1.0165  RMSE=9.719  MASE=0.261
[09:34:05]   Fold 4 | Year 2022  WAPE=0.5031  SMAPE=0.9164  RMSE=13.532  MASE=0.613
[09:34:08]   Fold 5 | Year 2023  WAPE=0.4504  SMAPE=0.8171  RMSE=19.896  MASE=0.628
[09:34:12]   Fold 6 | Year 2024  WAPE=0.5250  SMAPE=0.8983  RMSE=6.765  MASE=0.247
[09:34:12]   [GBR_log1p] → WAPE=0.5291  SMAPE=0.9109  MASE=0.5502  RMSE=19.584
[09:34:12]   [GBR_log1p] trial 2/12 params={'model__regressor__learning_rate': 0.06, 'model__regressor__max_depth': 2, 'mode

,model,WAPE,SMAPE,MASE,RMSE,params
0,XGB_log1p,0.506642,0.905720,0.525529,18.662915,{'model__regressor__colsample_bytree': 0.67186...
1,XGB_log1p,0.508698,0.906434,0.527388,18.689485,{'model__regressor__colsample_bytree': 0.97815...
2,XGB_log1p,0.508880,0.904784,0.527956,18.625963,{'model__regressor__colsample_bytree': 0.82305...
3,XGB_log1p,0.509353,0.903048,0.528283,18.642654,{'model__regressor__colsample_bytree': 0.85903...
4,XGB_log1p,0.509487,0.903761,0.528343,18.654944,{'model__regressor__colsample_bytree': 0.78740...
5,XGB_log1p,0.509973,0.903497,0.528976,18.709799,{'model__regressor__colsample_bytree': 0.78057...
6,XGB_log1p,0.510975,0.904103,0.529953,18.649340,{'model__regressor__colsample_bytree': 0.99489...
7,GBR_log1p,0.511537,0.921341,0.529052,18.585918,{'model__regressor__learning_rate': 0.02077244...
8,GBR_log1p,0.512078,0.930563,0.529642,18.559257,{'model__regressor__learning_rate': 0.01766628...
9,GBR_log1p,0.512414,0.916128,0.530239,18.674631,{'model__regressor__learning_rate': 0.02228953...


8) Train & compare — DEATHS hurdle (auto-select)

In [ ]:
# =========================
# DEATHS: hurdle training with interactive logs & retries
# =========================
import time, json, warnings, numpy as np, pandas as pd, joblib
from typing import Dict, Any, Tuple
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, roc_auc_score

np.random.seed(42)

def _safe_auc(y_true, y_prob):
    try:
        # avoid AUC when only one class in y_true for a fold
        if len(np.unique(y_true)) < 2: 
            return np.nan
        return float(roc_auc_score(y_true, y_prob))
    except Exception:
        return np.nan

def log(msg):
    ts = time.strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

def evaluate_deaths_hurdle(
    clf_pipe: Pipeline, clf_params: Dict[str, Any],
    reg_pipe: Pipeline, reg_params: Dict[str, Any],
    verbose: bool = True
) -> float:
    """
    Year-blocked rolling CV for hurdle model (classifier for zero/non-zero + regressor for positive counts).
    Logs per-fold prevalence, AUC (if available), and RMSE.
    Returns mean fold RMSE (lower is better).
    """
    clf_pipe.set_params(**clf_params)
    reg_pipe.set_params(**reg_params)

    fold_rmses, fold_aucs, fold_prev = [], [], []
    min_train_years = max(2, len(years_deaths)//2) if len(years_deaths) >= 3 else 1

    for tr_years, va_year in year_rolling_splits(years_deaths, min_train_years=min_train_years):
        tr = deaths_df[deaths_df["Year"].isin(tr_years)].copy()
        va = deaths_df[deaths_df["Year"] == va_year].copy()

        ytr = tr["Deaths"].values
        yva = va["Deaths"].values
        ytr_pos = (ytr > 0).astype(int)
        yva_pos = (yva > 0).astype(int)

        # ---- Classifier
        prep_c = clf_pipe.named_steps["prep"]
        model_c = clf_pipe.named_steps["model"]
        Xtr_c = prep_c.fit_transform(tr[feat_cols_d])
        Xva_c = prep_c.transform(va[feat_cols_d])

        # handle imbalance if model supports scale_pos_weight
        if hasattr(model_c, "get_params") and "scale_pos_weight" in model_c.get_params():
            pos = ytr_pos.sum(); neg = len(ytr_pos) - pos
            spw = float(max(1.0, neg / max(1, pos)))
            try:
                model_c.set_params(scale_pos_weight=spw)
            except Exception:
                pass

        name_c = type(model_c).__name__.lower()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if "xgb" in name_c:
                xgb_fit_with_eval(model_c, Xtr_c, ytr_pos, Xva_c, yva_pos, metrics=("logloss","auc"))
            elif "lgbm" in name_c:
                lgbm_fit_with_eval(model_c, Xtr_c, ytr_pos, Xva_c, yva_pos, metric="binary_logloss")
            elif "catboost" in name_c:
                cat_fit_with_eval(model_c, Xtr_c, ytr_pos, Xva_c, yva_pos, is_classifier=True)
            else:
                model_c.fit(Xtr_c, ytr_pos)

        # freeze fold model properly (avoid leaking pre-fit transforms)
        clf_fold = Pipeline([("prep", prep_c), ("model", model_c)])
        clf_fold.fit(tr[feat_cols_d], ytr_pos)

        # ---- Regressor (train only on positives)
        tr_pos = tr[ytr_pos == 1]
        if tr_pos.empty:
            # degenerate: predict zeros
            yh = np.zeros_like(yva, dtype=float)
            auc = np.nan
        else:
            prep_r = reg_pipe.named_steps["prep"]
            model_r = reg_pipe.named_steps["model"]
            Xtr_r = prep_r.fit_transform(tr_pos[feat_cols_d])
            Xva_r = prep_r.transform(va[feat_cols_d])

            name_r = type(model_r).__name__.lower()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                if "xgb" in name_r:
                    xgb_fit_with_eval(model_r, Xtr_r, tr_pos["Deaths"].values, Xva_r, yva, metrics=("rmse",))
                elif "lgbm" in name_r:
                    lgbm_fit_with_eval(model_r, Xtr_r, tr_pos["Deaths"].values, Xva_r, yva, metric="rmse")
                elif "catboost" in name_r:
                    cat_fit_with_eval(model_r, Xtr_r, tr_pos["Deaths"].values, Xva_r, yva, is_classifier=False)
                else:
                    model_r.fit(Xtr_r, tr_pos["Deaths"].values)

            reg_fold = Pipeline([("prep", prep_r), ("model", model_r)])
            reg_fold.fit(tr_pos[feat_cols_d], tr_pos["Deaths"].values)

            # predictions
            if hasattr(clf_fold.named_steps["model"], "predict_proba"):
                p_pos = clf_fold.predict_proba(va[feat_cols_d])[:, 1]
            else:
                # decision_function fallback → sigmoid
                z = clf_fold.named_steps["model"].decision_function(va[feat_cols_d])
                p_pos = 1.0 / (1.0 + np.exp(-z))

            mu_pos = np.clip(reg_fold.predict(va[feat_cols_d]), 0, None)
            yh = p_pos * mu_pos
            auc = _safe_auc(yva_pos, p_pos)

        rmse = float(np.sqrt(mean_squared_error(yva, yh)))
        prev = float(yva_pos.mean()) if len(yva_pos) else np.nan

        fold_rmses.append(rmse)
        fold_aucs.append(auc)
        fold_prev.append(prev)

        if verbose:
            log(f"Fold va_year={va_year} | prev={prev:.3f} | AUC={np.nan if np.isnan(auc) else round(auc,4)} | RMSE={rmse:.4f}")

    mean_rmse = float(np.nanmean(fold_rmses)) if len(fold_rmses) else np.inf
    if verbose:
        auc_txt = "n/a" if all(np.isnan(a) for a in fold_aucs) else f"{np.nanmean(fold_aucs):.4f}"
        log(f"→ Mean RMSE={mean_rmse:.4f} | Mean AUC={auc_txt} across {len(fold_rmses)} folds")
    return mean_rmse

# ---- Candidate spaces
cand_clf = deaths_classifier_candidates(feat_cols_d)
cand_reg = deaths_regressor_candidates(feat_cols_d)

# ---- Search + auto-retry until RMSE target or retries exhausted
TARGET_RMSE = 0.25      # adjust to your context; 0.25 is stringent for weekly deaths
MAX_RESTARTS = 2
TRIALS_EACH  = 5        # per restart; grows each restart

results_deaths_all = []
best_score = np.inf
best_combo = None

for restart in range(1, MAX_RESTARTS + 1):
    log(f"========== DEATHS SEARCH (restart {restart}/{MAX_RESTARTS}) ==========")
    tried = 0
    # iterate candidates; cap per-grid tries for runtime control
    for clf_name, (clf_pipe, clf_grid, _c_helper) in cand_clf.items():
        # randomize grid order for exploration
        grid_c = list(clf_grid)
        np.random.shuffle(grid_c)
        for clf_params in grid_c[:TRIALS_EACH]:
            for reg_name, (reg_pipe, reg_grid, _r_helper) in cand_reg.items():
                grid_r = list(reg_grid)
                np.random.shuffle(grid_r)
                for reg_params in grid_r[:TRIALS_EACH]:
                    tried += 1
                    log(f"Trial #{tried} → {clf_name} + {reg_name}")
                    score = evaluate_deaths_hurdle(clf_pipe, clf_params, reg_pipe, reg_params, verbose=True)
                    results_deaths_all.append({
                        "classifier": clf_name, "clf_params": clf_params,
                        "regressor": reg_name,  "reg_params": reg_params,
                        "RMSE": score
                    })
                    if score < best_score:
                        best_score = score
                        best_combo = (clf_name, clf_params, reg_name, reg_params)
                        log(f"🌟 NEW BEST: {clf_name} + {reg_name} RMSE={best_score:.4f}")

    log(f"[RESTART {restart}] Best so far RMSE={best_score:.4f}")
    if best_score <= TARGET_RMSE:
        log(f"✅ Target met (RMSE ≤ {TARGET_RMSE}). Stopping search.")
        break
    else:
        # widen the exploration next round
        TRIALS_EACH = min(TRIALS_EACH + 3, 12)
        log(f"🔁 Target not met; increasing trials per grid to {TRIALS_EACH} and retrying...")

# ---- Finalize & save
if best_combo is None:
    raise RuntimeError("No valid deaths model combination found.")

(clf_best_name, clf_best_params, reg_best_name, reg_best_params) = best_combo
log(f"[DEATHS] Winner: {clf_best_name} + {reg_best_name}  RMSE={best_score:.4f}")

# Rebuild best pipelines and refit on ALL deaths data
clf_final = deaths_classifier_candidates(feat_cols_d)[clf_best_name][0]
reg_final = deaths_regressor_candidates(feat_cols_d)[reg_best_name][0]
clf_final.set_params(**clf_best_params)
reg_final.set_params(**reg_best_params)

# Set scale_pos_weight on full fit if available
y_pos_all = (deaths_df["Deaths"] > 0).astype(int).values
if hasattr(clf_final.named_steps["model"], "get_params") and "scale_pos_weight" in clf_final.named_steps["model"].get_params():
    pos = y_pos_all.sum(); neg = len(y_pos_all) - pos
    spw = float(max(1.0, neg / max(1, pos)))
    try:
        clf_final.set_params(**{"model__scale_pos_weight": spw})
    except Exception:
        pass

clf_final.fit(deaths_df[feat_cols_d], y_pos_all)

dpos = deaths_df[y_pos_all == 1]
if dpos.empty:
    # guard: no positive deaths in the corpus
    reg_final.fit(deaths_df[feat_cols_d].iloc[:1], np.array([0.0]))
else:
    reg_final.fit(dpos[feat_cols_d], dpos["Deaths"].values)

# Save artifacts
joblib.dump(clf_final, os.path.join(OUTDIR, "best_deaths_stage1_classifier.joblib"))
joblib.dump(reg_final, os.path.join(OUTDIR, "best_deaths_stage2_regressor.joblib"))

# Summarize & persist a compact training report (append to existing if any)
try:
    # if you already wrote a training_summary.json for cases, read & extend
    summ_path = os.path.join(OUTDIR, "training_summary.json")
    try:
        with open(summ_path, "r", encoding="utf-8") as f:
            summary = json.load(f)
    except Exception:
        summary = {}

    dfd = pd.DataFrame(results_deaths_all).sort_values("RMSE")
    summary["deaths"] = {
        "winner_classifier": clf_best_name,
        "winner_regressor": reg_best_name,
        "metric": "RMSE",
        "score": float(best_score),
        "trials": int(len(dfd)),
        "target_rmse": float(TARGET_RMSE)
    }
    with open(summ_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    log("[SAVED] deaths models + updated training_summary.json")
except Exception as e:
    log(f"[WARN] Could not update training_summary.json: {e}")


[10:19:50] ========== DEATHS SEARCH (restart 1/2) ==========
[10:19:50] Trial #1 → LogisticRegression + PoissonRegressor


c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_glm\glm.py:283: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)


[10:19:51] Fold va_year=2019 | prev=0.004 | AUC=0.9885 | RMSE=0.0850


c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_glm\glm.py:283: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)


[10:19:52] Fold va_year=2020 | prev=0.011 | AUC=0.7416 | RMSE=0.1571


...

[10:40:20] Fold va_year=2022 | prev=0.010 | AUC=0.81 | RMSE=0.1265


c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_glm\glm.py:283: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)


[10:40:24] Fold va_year=2023 | prev=0.011 | AUC=0.8679 | RMSE=0.2067


c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_glm\glm.py:283: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)


[10:40:28] Fold va_year=2024 | prev=0.004 | AUC=0.9652 | RMSE=0.0751
[10:40:28] → Mean RMSE=0.1249 | Mean AUC=0.8610 across 6 folds
[10:40:28] Trial #55 → RandomForestClassifier + RandomForestRegressor
[10:40:34] Fold va_year=2019 | prev=0.004 | AUC=0.9635 | RMSE=0.0998
[10:40:40] Fold va_year=2020 | prev=0.011 | AUC=0.7476 | RMSE=0.1618
[10:40:45] Fold va_year=2021 | prev=0.005 | AUC=0.8116 | RMSE=0.1177
[10:40:51] Fold va_year=2022 | prev=0.010 | AUC=0.81 | RMSE=0.1296
[10:40:57] Fold va_year=2023 | prev=0.011 | AUC=0.8679 | RMSE=0.2004
[10:41:02] Fold va_year=2024 | prev=0.004 | AUC=0.9652 | RMSE=0.0810
[10:41:02] → Mean RMSE=0.1317 | Mean AUC=0.8610 across 6 folds
[10:41:02] Trial #56 → RandomForestClassifier + RandomForestRegressor
[10:41:04] Fold va_year=2019 | prev=0.004 | AUC=0.9635 | RMSE=0.0996
[10:41:06] Fold va_year=2020 | prev=0.011 | AUC=0.7476 | RMSE=0.1618
[10:41:09] Fold va_year=2021 | prev=0.005 | AUC=0.8116 | RMSE=0.1195
[10:41:12] Fold va_year=2022 | prev=0.010 | AU

In [10]:
import numpy as np
import json
import plotly.graph_objects as go

top_rows = dfd.nsmallest(20, "RMSE") if len(dfd) > 20 else dfd.copy()
labels = [f"{r['classifier']} + {r['regressor']}" for _, r in top_rows.iterrows()]

fig2 = go.Figure()
for idx, (_, r) in enumerate(top_rows.iterrows()):  # <- enumerate gives 0..n-1
    is_best = (
        (r["classifier"] == clf_best_name)
        and (r["regressor"] == reg_best_name)
        and np.isclose(r["RMSE"], best_score, rtol=1e-6, atol=1e-9)
    )
    fig2.add_bar(
        x=[labels[idx]],
        y=[r["RMSE"]],
        marker=dict(color="#ef4444" if is_best else "#94a3b8"),
        hovertext=[json.dumps(
            {"clf_params": r["clf_params"], "reg_params": r["reg_params"]},
            indent=2, default=str)],
        hoverinfo="text+y",
    )

fig2.update_layout(
    title="Deaths (Hurdle) — Model Combination Comparison (RMSE ↓)",
    xaxis_title="Classifier + Regressor",
    yaxis_title="RMSE",
    xaxis_tickangle=-30,
    showlegend=False,
)
fig2.show()


9) Final refit on ALL data & save artifacts

In [11]:
# --- CASES ---
best_pipe_cases = cases_candidates(feat_cols_c)[best_cases_name][0]
best_pipe_cases.set_params(**best_cases_params)
best_pipe_cases.fit(cases_df[feat_cols_c], cases_df["Cases"])
joblib.dump(best_pipe_cases, os.path.join(OUTDIR, "best_cases_pipeline.joblib"))

# --- DEATHS (hurdle) ---
clf_final = deaths_classifier_candidates(feat_cols_d)[clf_best_name][0]
reg_final = deaths_regressor_candidates(feat_cols_d)[reg_best_name][0]
clf_final.set_params(**clf_best_params)
reg_final.set_params(**reg_best_params)

# scale_pos_weight if available
y_pos_all = (deaths_df["Deaths"]>0).astype(int).values
if hasattr(clf_final.named_steps["model"], "set_params") and "scale_pos_weight" in clf_final.named_steps["model"].get_params():
    pos = y_pos_all.sum(); neg = len(y_pos_all)-pos
    spw = float(max(1.0, neg/max(1,pos)))
    clf_final.set_params(**{"model__scale_pos_weight": spw})

clf_final.fit(deaths_df[feat_cols_d], y_pos_all)
joblib.dump(clf_final, os.path.join(OUTDIR, "best_deaths_stage1_classifier.joblib"))

dpos = deaths_df[y_pos_all==1]
if dpos.empty:
    # no positives -> degenerate predictor
    # still save a trained regressor on minimal data to keep interface
    reg_final.fit(deaths_df[feat_cols_d].iloc[:1], np.array([0.0]))
else:
    reg_final.fit(dpos[feat_cols_d], dpos["Deaths"].values)
joblib.dump(reg_final, os.path.join(OUTDIR, "best_deaths_stage2_regressor.joblib"))

# --- Summary JSON for Streamlit/About tab ---
summary = {
    "cases": {
        "winner_model": best_cases_name,
        "winner_params": best_cases_params,
        "metric": "WAPE",
        "score": best_cases_wape,
        "trials": len(dfc)
    },
    "deaths": {
        "winner_classifier": clf_best_name,
        "winner_regressor": reg_best_name,
        "metric": "RMSE",
        "score": best_score,
        "trials": len(dfd)
    }
}
with open(os.path.join(OUTDIR, "training_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("[SAVED] cases/deaths models + training_summary.json")


c:\Users\Ovy\project\malaria_model\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning:

lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



[SAVED] cases/deaths models + training_summary.json


In [15]:
from sklearn.base import clone
import numpy as np
import pandas as pd
import warnings

# --- Cases: stability check via rolling-year CV and simple backtest ---
def _wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.maximum(1.0, np.abs(y_true))
    return float(np.mean(np.abs(y_true - y_pred) / denom))

def year_rolling_splits(years, min_train_years=2):
    years = sorted(list(years))
    for i in range(min_train_years, len(years)):
        yield years[:i], years[i]

# 1) Rolling-year validation WAPE
fold_scores = []
min_train_years = max(2, len(years_cases)//2) if len(years_cases) >= 3 else 1
for tr_years, va_year in year_rolling_splits(years_cases, min_train_years=min_train_years):
    tr = cases_df[cases_df["Year"].isin(tr_years)]
    va = cases_df[cases_df["Year"] == va_year]

    Xtr, ytr = tr[feat_cols_c], tr["Cases"].values
    Xva, yva = va[feat_cols_c], va["Cases"].values

    # fresh, unfitted copy each fold
    model_fold = clone(best_pipe_cases)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model_fold.fit(Xtr, ytr)

    yhat = np.clip(model_fold.predict(Xva), 0, None)
    fold_scores.append(_wape(yva, yhat))

cv_wape = float(np.nanmean(fold_scores)) if fold_scores else np.nan

# 2) Simple holdout by last year (if available)
holdout_wape = np.nan
if len(years_cases) >= 3:
    tr_years = years_cases[:-1]
    ho_year  = years_cases[-1]
    tr = cases_df[cases_df["Year"].isin(tr_years)]
    ho = cases_df[cases_df["Year"] == ho_year]

    Xtr, ytr = tr[feat_cols_c], tr["Cases"].values
    Xho, yho = ho[feat_cols_c], ho["Cases"].values

    model_ho = clone(best_pipe_cases)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model_ho.fit(Xtr, ytr)

    yhat = np.clip(model_ho.predict(Xho), 0, None)
    holdout_wape = _wape(yho, yhat)

print(f"[CASES] Rolling CV WAPE: {cv_wape:.4f} | Holdout WAPE: {holdout_wape:.4f}")


[CASES] Rolling CV WAPE: 1.8469 | Holdout WAPE: 1.2772


In [16]:
from sklearn.base import clone
import numpy as np
import warnings

# deaths_df, feat_cols_d, clf_final (stage1), reg_final (stage2) must already exist
# y_pos_all = (deaths_df["Deaths"] > 0).astype(int).values  # if not already defined
if 'y_pos_all' not in globals():
    y_pos_all = (deaths_df["Deaths"] > 0).astype(int).values

# Rolling-year CV on the hurdle pipeline (prob * reg)
rmse_scores = []
min_train_years_d = max(2, len(years_deaths)//2) if len(years_deaths) >= 3 else 1

for tr_years, va_year in year_rolling_splits(years_deaths, min_train_years=min_train_years_d):
    tr = deaths_df[deaths_df["Year"].isin(tr_years)]
    va = deaths_df[deaths_df["Year"] == va_year]

    Xtr, ytr = tr[feat_cols_d], tr["Deaths"].values
    Xva, yva = va[feat_cols_d], va["Deaths"].values

    # fresh clones each fold
    clf = clone(clf_final)
    reg = clone(reg_final)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        # stage 1: classifier on presence
        ytr_pos = (ytr > 0).astype(int)
        clf.fit(Xtr, ytr_pos)

        # stage 2: regressor on positive-only
        mask_pos = ytr > 0
        if mask_pos.any():
            reg.fit(Xtr[mask_pos], ytr[mask_pos])
        else:
            # degenerate fallback
            reg.fit(Xtr.iloc[:1], np.array([0.0]))

    # predict: prob * positive-mean
    if hasattr(clf, "predict_proba"):
        p = clf.predict_proba(Xva)[:, 1]
    else:
        z = clf.decision_function(Xva)
        p = 1.0 / (1.0 + np.exp(-z))

    mu = np.clip(reg.predict(Xva), 0, None)
    yhat = np.clip(p * mu, 0, None)

    rmse = float(np.sqrt(np.mean((yhat - yva) ** 2)))
    rmse_scores.append(rmse)

cv_rmse = float(np.nanmean(rmse_scores)) if rmse_scores else np.nan
print(f"[DEATHS] Rolling CV RMSE: {cv_rmse:.4f}")


[DEATHS] Rolling CV RMSE: 0.1171
